# STPA-based scenario generation

This notebook runs the **STPA** workflow in [Asago Scenario Generator](https://github.com/asago-ai/asago-scenario-generator). STPA (System-Theoretic Process Analysis) is a peer of the taxonomy/risk pipeline (see `taxonomy-risk-demo.ipynb`), not a compatibility mode.

The pipeline models **losses**, **hazards**, **control structure**, **unsafe control actions** (ICAs), and **enriched threats**, then produces scenarios, Gherkin specs, evaluation evidence, and an STPA HTML report.

**Stages:**
1. **SP1** — loss/hazard analysis + control structure (+ optional capability profile)
2. **SP2** — ICA enumeration + catalog enrichment (OWASP agentic / ATLAS)
3. **SP3** — scenario specs, narratives, attack trees, Gherkin, eval
4. **Report** — combined `stpa-report.html`

**What you'll see:**
1. Choose a bundled system (OcciAI, Klarna, or Airbnb)
2. Run `run_stpa_pipeline`
3. Inspect losses, hazards, control structure, ICAs, enriched threats, scenarios, Gherkin, eval, and the report
4. Optionally validate canonical Stage 6 execution projections when they are present

---

### Prerequisites

1. **Python virtual environment** — from the repo root:
   ```bash
   uv sync --extra scenario-generator
   ```
   In Jupyter, select this venv as your kernel: **Kernel → Change Kernel** → choose the `.venv` Python interpreter.

2. **A vLLM (or OpenAI-compatible) endpoint** serving `gemma-4-26b-a4b-it`.

STPA resolves the LLM client from environment variables when no named profile is passed (no `config/model-profiles.yaml` required):

```bash
export ASAGO_SCENARIO_GENERATOR_MODEL_BASE_URL="https://your-vllm-endpoint.com/v1"
export ASAGO_SCENARIO_GENERATOR_MODEL_NAME="gemma-4-26b-a4b-it"
export ASAGO_SCENARIO_GENERATOR_API_KEY="none"
export ASAGO_SCENARIO_GENERATOR_TEMPERATURE="1.0"
export ASAGO_SCENARIO_GENERATOR_MAX_COMPLETION_TOKENS="16384"
export ASAGO_SCENARIO_GENERATOR_TOP_P="0.95"
export ASAGO_SCENARIO_GENERATOR_TOP_K="64"
export ASAGO_SCENARIO_GENERATOR_USE_GUIDED_DECODING="false"
export ASAGO_SCENARIO_GENERATOR_RUN_LIVE="1"
```

The validated path uses `ASAGO_STPA_PROFILE=gemma4-oc` with `ASAGO_MODEL_PROFILES_FILE` pointing to an ignored local profiles file. Named profile values take precedence over environment sampling values.

To inspect an existing STPA output directory instead of calling the model, set `ASAGO_EXISTING_RUN_DIR` to a folder that contains `run-manifest.yaml` and `stpa-report.html` (for example a directory under the generator's `output/runs/`).


## 1. Configuration

Edit the values below if environment variables are not set.

> **Note:** STPA always runs the full SP1→SP3 workflow. The latest Klarna validation produced 12 scenarios in about three minutes on the configured OpenShift endpoint with two workers; runtime varies substantially by endpoint. Every live invocation uses a fresh timestamped output directory.


In [ ]:
import os

BASE_URL = os.environ.get("ASAGO_SCENARIO_GENERATOR_MODEL_BASE_URL", "http://localhost:8000/v1")
MODEL = os.environ.get("ASAGO_SCENARIO_GENERATOR_MODEL_NAME", "gemma-4-26b-a4b-it")
API_KEY = os.environ.get("ASAGO_SCENARIO_GENERATOR_API_KEY", "none")
TEMPERATURE = float(os.environ.get("ASAGO_SCENARIO_GENERATOR_TEMPERATURE", "1.0"))
MAX_COMPLETION_TOKENS = int(os.environ.get("ASAGO_SCENARIO_GENERATOR_MAX_COMPLETION_TOKENS", "16384"))
TOP_P = float(os.environ.get("ASAGO_SCENARIO_GENERATOR_TOP_P", "0.95"))
TOP_K = int(os.environ.get("ASAGO_SCENARIO_GENERATOR_TOP_K", "64"))
_guided = os.environ.get("ASAGO_SCENARIO_GENERATOR_USE_GUIDED_DECODING", "false").lower()
if _guided not in {"1", "true", "yes", "on", "0", "false", "no", "off"}:
    raise ValueError("ASAGO_SCENARIO_GENERATOR_USE_GUIDED_DECODING must be a boolean")
USE_GUIDED_DECODING = _guided in {"1", "true", "yes", "on"}
RUN_LIVE = os.environ.get("ASAGO_SCENARIO_GENERATOR_RUN_LIVE", "").lower() in {"1", "true", "yes"}
EXISTING_RUN_DIR = os.environ.get("ASAGO_EXISTING_RUN_DIR", "").strip()
MAX_WORKERS = int(os.environ.get("ASAGO_STPA_MAX_WORKERS", "2"))
# Named profile in config/model-profiles.yaml; leave empty to use env vars.
STPA_PROFILE = os.environ.get("ASAGO_STPA_PROFILE", "").strip() or None
PROFILES_FILE = os.environ.get("ASAGO_MODEL_PROFILES_FILE", "config/model-profiles.yaml")

for thread_var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(thread_var, "1")

if not STPA_PROFILE:
    os.environ["ASAGO_SCENARIO_GENERATOR_MODEL_BASE_URL"] = BASE_URL
    os.environ["ASAGO_SCENARIO_GENERATOR_MODEL_NAME"] = MODEL
    os.environ["ASAGO_SCENARIO_GENERATOR_API_KEY"] = API_KEY
    os.environ["ASAGO_SCENARIO_GENERATOR_TEMPERATURE"] = str(TEMPERATURE)
    os.environ["ASAGO_SCENARIO_GENERATOR_MAX_COMPLETION_TOKENS"] = str(MAX_COMPLETION_TOKENS)
    os.environ["ASAGO_SCENARIO_GENERATOR_TOP_P"] = str(TOP_P)
    os.environ["ASAGO_SCENARIO_GENERATOR_TOP_K"] = str(TOP_K)
    os.environ["ASAGO_SCENARIO_GENERATOR_USE_GUIDED_DECODING"] = str(USE_GUIDED_DECODING).lower()

print(f"Model:        {MODEL}")
if not STPA_PROFILE:
    print(f"Sampling:     temperature={TEMPERATURE}, max_tokens={MAX_COMPLETION_TOKENS}, top_p={TOP_P}, top_k={TOP_K}, guided={USE_GUIDED_DECODING}")
print(f"Live calls:   {RUN_LIVE}")
print(f"Profile:      {STPA_PROFILE or '(environment variables)'}")
print(f"Profiles file:{PROFILES_FILE if STPA_PROFILE else '(not used)'}")
print(f"Max workers:  {MAX_WORKERS}")
print(f"Existing run: {EXISTING_RUN_DIR or '(none — will generate)'}")


In [ ]:
from pathlib import Path
import os
import json

import ipywidgets as widgets
import pandas as pd
import yaml
from IPython.display import display, HTML, Markdown


def find_inputs_dir() -> Path:
    here = Path.cwd().resolve()
    candidates = [
        here / "inputs",
        here / "asago-scenario-generator" / "inputs",
        here.parent / "asago-scenario-generator" / "inputs",
    ]
    for candidate in candidates:
        if candidate.is_dir() and (candidate / "use-cases").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find asago-scenario-generator/inputs. "
        "Run Jupyter from the repo root or the asago-scenario-generator folder."
    )


INPUTS = find_inputs_dir()
OUTPUT_ROOT = INPUTS.parent / "output"
OUTPUT_ROOT.mkdir(exist_ok=True)

SYSTEMS = {
    "Klarna (fintech CS agent)": {
        "use_case": INPUTS / "use-cases" / "use-case-klarna-fs-isac-v36.txt",
        "risk_extraction": INPUTS / "risk-extractions" / "risk-extraction-fs-isac.json",
        "profile": INPUTS / "profiles" / "klarna-capability-profile.yaml",
        "qualification_facts": INPUTS / "profiles" / "klarna-qualification-facts.yaml",
        "policy": "FS-ISAC generative AI policy",
    },
    "OcciAI (NHS patient portal)": {
        "use_case": INPUTS / "use-cases" / "use-case-occiAI-guy-nhs-v1.txt",
        "risk_extraction": INPUTS / "risk-extractions" / "risk-extraction-guy-nhs.json",
        "profile": None,
        "qualification_facts": None,
        "policy": "Guy's and St Thomas' NHS AI policy",
    },
    "Airbnb (travel support platform)": {
        "use_case": INPUTS / "use-cases" / "uc-airbnb-conversational-ai-platform.md",
        "risk_extraction": INPUTS / "risk-extractions" / "risk-extraction-amadeus.json",
        "profile": None,
        "qualification_facts": None,
        "policy": "Amadeus responsible AI policy",
    },
}

SSSOM_PATH = INPUTS / "mappings" / "risk_to_category.sssom.tsv"

print(f"Inputs:  {INPUTS}")
print(f"Output:  {OUTPUT_ROOT}")


## 2. Select a system

STPA consumes the same use-case + risk-extraction pairs as the taxonomy/risk notebook. SSSOM mappings are **not** required here — catalog correspondence happens during SP2 enrichment.


In [ ]:
system_dropdown = widgets.Dropdown(
    options=list(SYSTEMS.keys()),
    value="Klarna (fintech CS agent)",
    description="System:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="520px"),
)
display(system_dropdown)


def selected_system() -> dict:
    return SYSTEMS[system_dropdown.value]


## 3. Preview inputs


In [ ]:
from asago_scenario_generator.data.loaders import load_risk_extraction

sys = selected_system()
use_case_text = sys["use_case"].read_text(encoding="utf-8")
risk_cards = load_risk_extraction(sys["risk_extraction"])

print(f"System:           {system_dropdown.value}")
print(f"Policy source:    {sys['policy']}")
print(f"Use case:         {sys['use_case'].name} ({len(use_case_text):,} chars)")
print(f"Risk extraction:  {sys['risk_extraction'].name}")
print(f"Atlas risk cards: {len(risk_cards)}")
print()
print("Use-case excerpt:")
print("-" * 60)
print("\n".join(use_case_text.strip().splitlines()[:18]))
print("...")


## 4. Run the STPA pipeline

This calls `asago_scenario_generator.stpa.pipeline.run_stpa_pipeline`, the same entry point as:

```bash
uv run asago-scenario-generator stpa-run \
  --use-case use-case.txt \
  --risk-extraction risk-extraction.json \
  --output-dir output/stpa \
  --max-workers 2
```

Unlike taxonomy/risk generation, STPA writes into the directory you pass (it does not create a hashed child run id). Use a fresh output folder per live run, or pass `resume=True` to skip stages whose artefacts already exist.


In [ ]:
import inspect
from contextlib import contextmanager
from datetime import datetime, timezone
import threading
import time

from asago_scenario_generator.stpa.pipeline import run_stpa_pipeline

@contextmanager
def progress_heartbeat(label: str, interval_seconds: int = 60):
    stop = threading.Event()
    started = time.monotonic()

    def report_progress():
        while not stop.wait(interval_seconds):
            elapsed_minutes = int((time.monotonic() - started) // 60)
            print(f"{label}: still running ({elapsed_minutes} minute(s) elapsed)", flush=True)

    thread = threading.Thread(target=report_progress, daemon=True)
    thread.start()
    try:
        yield
    finally:
        stop.set()
        thread.join(timeout=1)

required_stpa_parameters = {"profiles_file", "max_workers", "resume"}
missing_stpa_parameters = required_stpa_parameters - set(inspect.signature(run_stpa_pipeline).parameters)
if missing_stpa_parameters and not EXISTING_RUN_DIR:
    raise RuntimeError(
        "The installed asago-scenario-generator is too old for this example; "
        f"run_stpa_pipeline is missing: {sorted(missing_stpa_parameters)}."
    )

sys = selected_system()
slug = (
    "occiai" if "OcciAI" in system_dropdown.value
    else "klarna" if "Klarna" in system_dropdown.value
    else "airbnb"
)

if EXISTING_RUN_DIR:
    run_dir = Path(EXISTING_RUN_DIR)
    if not run_dir.exists():
        raise FileNotFoundError(run_dir)
    print(f"Skipping live generation; inspecting {run_dir}")
    stpa_result = None
else:
    if not RUN_LIVE:
        raise RuntimeError(
            "Live model calls are disabled. Set ASAGO_SCENARIO_GENERATOR_RUN_LIVE=1 "
            "or set ASAGO_EXISTING_RUN_DIR to inspect an existing run."
        )
    if STPA_PROFILE and not Path(PROFILES_FILE).is_file():
        raise FileNotFoundError(f"Model profiles file not found: {PROFILES_FILE}")
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = OUTPUT_ROOT / "stpa" / f"{slug}-{run_stamp}"
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"Running STPA for: {system_dropdown.value}")
    print(f"Output dir: {run_dir}")
    with progress_heartbeat("STPA pipeline"):
        stpa_result = run_stpa_pipeline(
            use_case_path=str(sys["use_case"]),
            risk_extraction_path=str(sys["risk_extraction"]),
            output_dir=run_dir,
            profile=STPA_PROFILE,
            profiles_file=PROFILES_FILE,
            max_workers=MAX_WORKERS,
            resume=False,
        )
    print()
    print("STPA pipeline returned.")
    if stpa_result.stage_errors:
        print("Stage errors:")
        for err in stpa_result.stage_errors:
            print(f"  - {err}")
    else:
        print("No stage errors recorded.")
    if stpa_result.stage_warnings:
        print(f"Recoverable stage warnings: {len(stpa_result.stage_warnings)}")
        for warning in stpa_result.stage_warnings:
            print(f"  - {warning}")
    else:
        print("No recoverable stage warnings recorded.")
    post_revision_warnings = (
        stpa_result.sp1_result.post_revision_warnings
        if stpa_result.sp1_result is not None
        else []
    )
    if post_revision_warnings:
        print(f"SP1 post-revision validation warnings: {len(post_revision_warnings)}")
        for warning in post_revision_warnings:
            print(f"  - {warning}")
    if stpa_result.sp3_result is not None:
        sp3 = stpa_result.sp3_result
        print(f"  Scenario specs:     {len(sp3.scenario_specs)}")
        print(f"  Scenario envelopes: {len(sp3.scenario_envelopes)}")
        print(f"  Validation errors:  {len(sp3.validation_errors)}")
    if stpa_result.report_path:
        print(f"  Report:             {stpa_result.report_path}")
    if stpa_result.stage_errors:
        raise RuntimeError(f"STPA completed with stage errors: {stpa_result.stage_errors}")
    if stpa_result.sp3_result is None or not stpa_result.sp3_result.scenario_envelopes:
        raise RuntimeError("STPA completed without scenario envelopes.")
    if stpa_result.sp3_result.validation_errors:
        raise RuntimeError("STPA completed with validation errors.")
    if not stpa_result.report_path or not Path(stpa_result.report_path).is_file():
        raise RuntimeError("STPA report was not generated.")


## 5. SP1 — Losses and hazards

Losses come from risk cards and from the use case. Hazards are system conditions that can lead to those losses; security constraints bound them.


In [ ]:
loss_path = run_dir / "loss-analysis.yaml"
loss = yaml.safe_load(loss_path.read_text(encoding="utf-8"))

risk_losses = loss.get("risk_card_losses") or []
use_case_losses = loss.get("use_case_losses") or []
hazards = loss.get("hazards") or []
constraints = loss.get("security_constraints") or []

print(f"Risk-card losses: {len(risk_losses)}")
print(f"Use-case losses:  {len(use_case_losses)}")
print(f"Hazards:          {len(hazards)}")
print(f"Constraints:      {len(constraints)}")

loss_rows = [
    {
        "ID": item.get("loss_id"),
        "Provenance": item.get("provenance"),
        "Description": item.get("description"),
        "Source risks": ", ".join(item.get("source_risk_cards") or []),
    }
    for item in risk_losses + use_case_losses
]
display(pd.DataFrame(loss_rows))

hazard_rows = [
    {
        "ID": h.get("hazard_id"),
        "Related losses": ", ".join(h.get("related_losses") or []),
        "Description": h.get("description"),
    }
    for h in hazards
]
print()
display(pd.DataFrame(hazard_rows))


## 6. SP1 — Control structure

Responsibilities, control actions, process-model beliefs, and feedback loops. Later ICA identifiers (`RESP-*:CA-*:TYPE`) and causal factors (`PM-*`, `FB-*`, `CA-*`) refer into this structure.


In [ ]:
cs = yaml.safe_load((run_dir / "control-structure.yaml").read_text(encoding="utf-8"))
responsibilities = cs.get("responsibilities") or []
processes = cs.get("controlled_processes") or []
links = cs.get("coordination_links") or []

print(f"Responsibilities:     {len(responsibilities)}")
print(f"Controlled processes: {len(processes)}")
print(f"Coordination links:   {len(links)}")

resp_rows = []
for resp in responsibilities:
    actions = resp.get("control_actions") or []
    beliefs = resp.get("process_model_parts") or []
    resp_rows.append({
        "ID": resp.get("resp_id"),
        "Description": (resp.get("description") or "")[:100],
        "Control actions": len(actions),
        "Process-model parts": len(beliefs),
        "Action IDs": ", ".join(a.get("ca_id", "") for a in actions),
    })
display(pd.DataFrame(resp_rows))


## 7. SP2 — Unsafe control actions (ICA enumeration)

Each control action is considered under four UCA types: not provided, incorrect, wrong timing, wrong duration. Some slots are marked N/A with a justification.


In [ ]:
ica = yaml.safe_load((run_dir / "ica-enumeration.yaml").read_text(encoding="utf-8"))
slots = ica.get("slots") or []
na_slots = [s for s in slots if s.get("is_na")]
active_slots = [s for s in slots if not s.get("is_na")]
ica_count = sum(len(s.get("icas") or []) for s in active_slots)

print(f"Slots:        {len(slots)}")
print(f"Active:       {len(active_slots)}")
print(f"N/A:          {len(na_slots)}")
print(f"ICA instances:{ica_count}")

slot_rows = []
for slot in slots:
    slot_rows.append({
        "Slot": slot.get("slot_id"),
        "UCA type": slot.get("uca_type"),
        "N/A": slot.get("is_na"),
        "ICAs": len(slot.get("icas") or []),
        "Control action": slot.get("control_action"),
        "Responsibility": slot.get("responsibility") or slot.get("coordination_link") or "",
    })
display(pd.DataFrame(slot_rows).head(30))

print()
print("Sample ICAs:")
shown = 0
for slot in active_slots:
    for inst in slot.get("icas") or []:
        print(f"  {inst.get('ica_id')}: {inst.get('ica_text', '')[:160]}")
        shown += 1
        if shown >= 8:
            break
    if shown >= 8:
        break


## 8. SP2 — Enriched threats

Structural threats (from ICAs) are mapped onto OWASP agentic / ATLAS / ASI catalog entries where correspondence exists.


In [ ]:
enriched = yaml.safe_load((run_dir / "enriched-threats.yaml").read_text(encoding="utf-8"))
threats = enriched.get("structural_threats") or []
coverage = enriched.get("coverage_analysis") or {}

print(f"Structural threats: {len(threats)}")
sc = coverage.get("structural_coverage") or {}
if sc:
    print(
        f"Structural coverage: {sc.get('non_na')}/{sc.get('total_slots')} "
        f"(rate={sc.get('coverage_rate')})"
    )
print(f"Uncovered OWASP: {coverage.get('uncovered_owasp_threats')}")

trows = []
for t in threats:
    mappings = t.get("catalog_mappings") or []
    trows.append({
        "ICA": t.get("ica_id") or t.get("ica_slot_id"),
        "Hazards": ", ".join(t.get("related_hazards") or []),
        "Catalog": ", ".join(f"{m.get('catalog')}:{m.get('id')}" for m in mappings),
        "Text": (t.get("ica_text") or "")[:120],
    })
display(pd.DataFrame(trows).head(20))


## 9. SP3 — Scenarios and Gherkin

Each scenario traces to an ICA slot (`RESP:CA:UCA-type`) and includes defender/attacker BDI, a loss scenario, narrative, and a `.feature` file.


In [ ]:
from IPython.display import display, HTML

scenario_dir = run_dir / "scenarios"
yaml_files = sorted(p for p in scenario_dir.glob("SCN-*.yaml")) if scenario_dir.exists() else []
feature_files = {p.stem: p for p in scenario_dir.glob("SCN-*.feature")} if scenario_dir.exists() else {}

print(f"Scenario YAML: {len(yaml_files)}")
print(f"Gherkin files: {len(feature_files)}")

rows = []
loaded = []
for path in yaml_files:
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    spec = data.get("scenario_spec") or {}
    source = spec.get("threat_source") or {}
    loaded.append((path, data))
    rows.append({
        "ID": data.get("scenario_id") or spec.get("scenario_id") or path.stem,
        "Controller": spec.get("target_controller"),
        "Control action": spec.get("target_control_action"),
        "UCA type": spec.get("ica_type"),
        "ICA": source.get("ica_id") or source.get("ica_slot_id"),
        "Gherkin": path.stem in feature_files,
    })
if rows:
    display(pd.DataFrame(rows))
else:
    print("No SCN-*.yaml files found. Check stage_errors and run-manifest.yaml.")


In [ ]:
def render_stpa_scenario(data: dict, feature_text: str | None, index: int) -> str:
    spec = data.get("scenario_spec") or {}
    narrative = data.get("narrative") or ""
    title = data.get("scenario_id") or spec.get("scenario_id") or f"scenario {index + 1}"
    html = '<div style="border:1px solid #ddd; padding:16px; margin:8px 0; border-radius:8px; background:#fafafa;">'
    html += f'<h3 style="margin-top:0;">{title}</h3>'
    html += (
        f'<p><b>Controller:</b> {spec.get("target_controller")} &nbsp; '
        f'<b>Action:</b> {spec.get("target_control_action")} &nbsp; '
        f'<b>UCA:</b> {spec.get("ica_type")}</p>'
    )
    if spec.get("loss_scenario"):
        html += f'<p><b>Loss scenario:</b> {spec["loss_scenario"]}</p>'
    attacker = spec.get("attacker_bdi") or {}
    if attacker.get("desires"):
        html += "<p><b>Attacker desire:</b> " + "; ".join(attacker["desires"][:2]) + "</p>"
    if narrative:
        excerpt = narrative if len(narrative) < 1200 else narrative[:1200] + "…"
        html += f'<div style="background:#e8f4f8; padding:8px; border-left:3px solid #2196F3; font-size:0.9em; white-space:pre-wrap;">{excerpt}</div>'
    if feature_text:
        html += "<h4>Gherkin</h4>"
        html += f'<pre style="background:#fff; padding:8px; font-size:0.85em; overflow:auto;">{feature_text}</pre>'
    html += "</div>"
    return html


for i, (path, data) in enumerate(loaded[:6]):
    feature = feature_files.get(path.stem)
    feature_text = feature.read_text(encoding="utf-8") if feature else None
    display(HTML(render_stpa_scenario(data, feature_text, i)))
if len(loaded) > 6:
    print(f"... and {len(loaded) - 6} more scenarios in {scenario_dir}")


## 10. Eval scorecard

Deterministic STPA metrics: structural consideration, BDI grounding, tree-branch coverage, traceability, diversity, and coverage gaps.


In [ ]:
scorecard_path = run_dir / "eval-scorecard.yaml"
if scorecard_path.exists():
    scorecard = yaml.safe_load(scorecard_path.read_text(encoding="utf-8"))
    metrics = scorecard.get("metrics") or scorecard
    for name, payload in (metrics.items() if isinstance(metrics, dict) else []):
        if name in {"coverage_gaps"}:
            continue
        print(f"{name}:")
        if isinstance(payload, dict):
            for k, v in payload.items():
                if not isinstance(v, dict):
                    print(f"  {k}: {v}")
        else:
            print(f"  {payload}")
        print()
    gaps = scorecard.get("coverage_gaps") or (metrics.get("coverage_gaps") if isinstance(metrics, dict) else None)
    if gaps:
        print("Coverage gaps — uncovered OWASP:", gaps.get("uncovered_owasp_threats"))
        print("Orphan elements:", gaps.get("orphan_elements"))
else:
    print("No eval-scorecard.yaml in this run.")


## 11. Execution projections (when present)

Newer STPA builds export canonical `stpa-execution-projection-v1` documents (JSON/YAML) with stable `EXEC:<controller>:<control-action>:<uca-type>` identities. The public validator checks traceability without reconstructing project objects.


In [ ]:
try:
    from asago_scenario_generator.stpa.scenario_prod.projection import (
        validate_exported_projection,
    )
except ImportError:
    validate_exported_projection = None
    print("This package build does not export validate_exported_projection; skipping.")

projection_files = []
if (run_dir / "scenarios" / "canonical").exists():
    projection_files.extend(sorted((run_dir / "scenarios" / "canonical").glob("*.json")))
    projection_files.extend(sorted((run_dir / "scenarios" / "canonical").glob("*.yaml")))
projection_files.extend(sorted(run_dir.glob("**/*projection*.json")))
projection_files.extend(sorted(run_dir.glob("**/*projection*.yaml")))
seen = set()
unique = []
for path in projection_files:
    resolved = path.resolve()
    if resolved not in seen:
        seen.add(resolved)
        unique.append(path)

print(f"Projection artefacts: {len(unique)}")
if validate_exported_projection is None:
    if unique:
        print("Upgrade asago-scenario-generator to validate stpa-execution-projection-v1 files.")
elif not unique:
    print(
        "This run has no standalone projection exports. "
        "That is expected for older STPA output; generate a new run to emit "
        "stpa-execution-projection-v1 artefacts."
    )
else:
    for path in unique[:10]:
        suffix = path.suffix.lower()
        payload = json.loads(path.read_text(encoding="utf-8")) if suffix == ".json" else yaml.safe_load(path.read_text(encoding="utf-8"))
        if not isinstance(payload, dict):
            print(f"  {path.name}: skipped (not an object)")
            continue
        try:
            check = validate_exported_projection(payload)
        except Exception as exc:
            print(f"  {path.name}: could not validate ({exc})")
            continue
        status = "valid" if check.valid else f"invalid ({len(check.violations)} violations)"
        try:
            rel = path.relative_to(run_dir)
        except ValueError:
            rel = path.name
        print(f"  {rel}: {status}")
        if not check.valid:
            for violation in check.violations[:5]:
                print(f"    - {violation.code}: {violation.detail} ({violation.element_id})")


## 12. STPA report and run manifest

`asago-scenario-generator stpa-report --output-dir <dir>` regenerates the HTML report from the artefacts already on disk.


In [ ]:
manifest_path = run_dir / "run-manifest.yaml"
if manifest_path.exists():
    manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
    manifest_stage_errors = manifest.get("stage_errors") or []
    manifest_stage_warnings = manifest.get("stage_warnings") or []
    manifest_post_revision_warnings = manifest.get("post_revision_warnings") or []
    print(f"Run ID:          {manifest.get('run_id')}")
    print(f"Scenario count:  {manifest.get('scenario_count')}")
    print(f"Stage errors:    {len(manifest_stage_errors)}")
    print(f"Stage warnings:  {len(manifest_stage_warnings)}")
    print(f"Post-revision warnings: {len(manifest_post_revision_warnings)}")
    summary = manifest.get("stage_summary") or {}
    if summary:
        print()
        print("Stage summary:")
        for stage, stats in summary.items():
            print(f"  {stage}: calls={stats.get('call_count')} tokens={stats.get('total_tokens')}")
    calls_path = run_dir / "calls.jsonl"
    if calls_path.exists():
        call_entries = [json.loads(line) for line in calls_path.read_text(encoding="utf-8").splitlines() if line]
        failed_calls = [entry for entry in call_entries if entry.get("success") is False]
        print()
        print(f"Model calls:      {len(call_entries)} ({len(failed_calls)} recoverable failures)")
        for entry in failed_calls:
            error_type = (entry.get("error") or "unknown error").splitlines()[0]
            print(f"  - {entry.get('stage')}/{entry.get('step')}: {error_type}")

report_path = run_dir / "stpa-report.html"
print()
if report_path.exists():
    print(f"STPA report: {report_path}")
    print("Open that file in a browser to review losses, ICAs, scenarios, and eval together.")
else:
    print("No stpa-report.html in this directory.")
print(f"Run directory: {run_dir}")
